In [37]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from pydantic import BaseModel,Field
from  langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
from langgraph.checkpoint.memory import InMemorySaver


In [38]:
class joke(BaseModel):
    joke:str=Field(...,description="A short joke to be told ")
    joke_exp:str=Field(...,description="A short explanation of the joke")
    

In [39]:
model=ChatOpenAI()
joke_model=model.with_structured_output(joke)

d:\AgenticAI_Learning\LangGraph\venv\Lib\site-packages\langchain_openai\chat_models\base.py:2418: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [40]:
class JokeState(TypedDict):
    topic:str
    joke:str
    joke_explanation:str

In [41]:
def gen(state:JokeState)->JokeState:
    joke_out=joke_model.invoke(f"Tell me a joke about {state['topic']} in short ")
    state['joke']=joke_out.joke
    return state

def exp(state:JokeState)->JokeState:
    joke_out=joke_model.invoke(f"Explain the following joke in short: {state['joke']}")
    state['joke_explanation']=joke_out.joke_exp
    return state

In [42]:
graph=StateGraph(JokeState)

graph.add_node("Generate Joke",gen)
graph.add_node("Explain Joke",exp)

graph.add_edge(START,"Generate Joke")
graph.add_edge("Generate Joke","Explain Joke")
graph.add_edge("Explain Joke",END)


checkpointer=InMemorySaver()


workflow=graph.compile(checkpointer=checkpointer)


In [43]:
config1={"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':"programming"},config=config1)

{'topic': 'programming',
 'joke': 'Why do programmers prefer dark mode?',
 'joke_explanation': 'Because light attracts bugs!'}

In [ ]:
workflow.get_state(config1)
for state in workflow.get_state(config1):
    print(state)



StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?', 'joke_explanation': 'Because light attracts bugs!'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1715b0-5e11-6d9a-8002-8d6e637c896a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-26T12:31:57.979791+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1715b0-49f7-66f4-8001-fbf41bed3cf3'}}, tasks=(), interrupts=())